- ### Error handle

1. [Try–catch](#trycatch)

2. [Throw exception](#throw-exception)

3. [Expected](#expected)

---

- ### try–catch

In [2]:
# Setup for oneline command %%cpp
import os, tempfile, subprocess
from IPython.core.magic import register_cell_magic
import shlex

@register_cell_magic
def cpp(line, cell):
    """
    Usage:
    %%cpp -i "input for cin" -- arg1 arg2 ...
    """
    tokens = shlex.split(line)
    input_data = None
    run_args = []

    # Parse stdin input
    if "-i" in tokens:
        idx = tokens.index("-i")
        if idx + 1 < len(tokens):
            input_data = tokens[idx + 1]

    # Parse program arguments after --
    if "--" in tokens:
        idx = tokens.index("--")
        run_args = tokens[idx + 1:]

    # Write temp C++ file
    with tempfile.NamedTemporaryFile(suffix=".cpp", delete=False, mode="w") as tmp_cpp:
        tmp_cpp.write(cell)
        cpp_path = tmp_cpp.name
    exe_path = cpp_path[:-4] + ".exe"

    try:
        # Compile
        compile_proc = subprocess.run(
            ["g++", "-std=c++23", "-O2", "-Wall", cpp_path, "-o", exe_path],
            capture_output=True,
            text=True
        )
        if compile_proc.returncode != 0:
            print("❌ Compilation failed:\n", compile_proc.stderr)
            return

        # Run program
        run_proc = subprocess.run(
            [exe_path] + run_args,
            input=input_data,      # feed stdin here
            capture_output=True,
            text=True
        )
        if run_proc.stdout:
            print(run_proc.stdout, end="")
        if run_proc.stderr:
            print("⚠️ Runtime error:\n", run_proc.stderr)

    finally:
        for f in (cpp_path, exe_path):
            try: os.remove(f)
            except: pass

In [28]:
%%cpp
#include <iostream>
using namespace std;

int main() {
    try {
        string s("hello");
        cout << stoi(s) << endl;
    }
    catch (const exception& e) {
        cout << "Error:" << e.what() << endl;
    }
}

Error:stoi


In [29]:
%%cpp
#include <iostream>
using namespace std;

int main() {
    try {
        string s("hello");
        cout << stoi(s) << endl;
    }
    catch (...) {
        cout << "Unkown Error" << endl;
    }
}

Unkown Error


---

- ### Throw exception

In [33]:
%%cpp
#include <iostream>
using namespace std;

int main() {
    try {
        string s("hello");
        cout << stoi(s) << endl;
    }
    catch (const exception& e) {
        cout << "Error:" << e.what() << endl;
    }
    throw runtime_error("Something went wrong");
}

Error:stoi
⚠️ Runtime error:
 terminate called after throwing an instance of 'std::runtime_error'
  what():  Something went wrong



---

- ### Expected

In [36]:
%%cpp
#include <iostream>
#include <string>
#include <expected>
using namespace std;

expected<int, string> divide(int a, int b) {
    if (b == 0) {
        return unexpected<string>("Division by zero");
    }
    return a / b;
}

int main() {
    auto result = divide(10, 0);
    if (!result) {
        cout << "Error: " << result.error() << endl;
    } else {
        cout << "Result: " << *result << endl;
    }
}

Error: Division by zero
